# Treinamento YOLOv8n — Dataset epi-v1
### Aula 6 — Atividade 2 (Intensivo IA Cariri / PNAAT)

Projeto: [yolo-edge-api](https://github.com/kaypes/yolo-edge-api) — João Kayque Pereira de Souza

Este notebook treina um **YOLOv8n** no dataset próprio `epi-v1` (Capacete, Colete, Pessoa),
construído, anotado e versionado nas Aulas 4 e 5 do curso. O resultado (`best.pt`) substitui o
`yolov8n.pt` genérico do COCO usado em produção desde a Aula 2 -- é o primeiro modelo do projeto
efetivamente treinado nas próprias classes de EPI.

**Antes de rodar:** em `Ambiente de execução → Alterar o tipo de ambiente de execução`,
selecione **GPU (T4)**. Sem GPU, 50 épocas em CPU podem levar horas em vez de ~25 minutos.


## 1. Patch do `torch.load` e confirmação da GPU

Versões do PyTorch 2.6+ mudaram o padrão de `weights_only` para `True`, o que bloqueia o
carregamento de checkpoints do Ultralytics como `yolov8n.pt` -- o mesmo patch já usado em
`app/model.py`, `scripts/validate_model.py` e `preprocessing/utils/evaluate.py` no repositório.

In [ ]:
!pip install -q ultralytics roboflow

import torch

_orig_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)

torch.load = _patched_torch_load

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Baixando o dataset `epi-v1` do Roboflow

O dataset já existe, versionado no Roboflow (workspace `kayque-pereira`,
projeto `epi-detection-rpi5-desz2-hwmyf`, versão 1). A chave de API é digitada de forma oculta
(`getpass`) em vez de hardcoded no notebook, já que este link vai ser compartilhado.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass("Cole sua API key do Roboflow: ")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("kayque-pereira").project("epi-detection-rpi5-desz2-hwmyf")
dataset = project.version(1).download("yolov8")

print("Dataset baixado em:", dataset.location)


### Alternativa: upload manual (fallback)

Caso o download acima falhe (chave de API expirada, etc.), suba o zip do dataset
(`dataset/exports/epi-v1`, já exportado e versionado no DVC do projeto) manualmente pelo painel
de arquivos do Colab e descompacte com a célula abaixo -- **pule esta célula se a anterior já
funcionou**.

In [ ]:
# import zipfile
# with zipfile.ZipFile("epi-v1.zip", "r") as z:
#     z.extractall("epi-v1")
#
# class _Dataset:
#     location = "epi-v1"
# dataset = _Dataset()


## 3. Ajustando e conferindo o `data.yaml`

O Roboflow exporta o `data.yaml` com um `path:` relativo ao ambiente onde foi gerado -- aqui
reescrevemos para o diretório do Colab e conferimos os três splits e as classes.

### Entregável: `dataset.yaml` com caminhos de train/val e lista de classes

In [ ]:
import yaml
from pathlib import Path

data_yaml_path = Path(dataset.location) / "data.yaml"

with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(Path(dataset.location).resolve())

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

print(data_yaml_path.read_text())


## 4. Treinando o modelo

`device=0` usa a GPU do Colab. `epochs=50` conforme exigido pela rubrica da atividade;
`patience=50` (maior que o número de épocas) desativa o early stopping para garantir que as
50 épocas completas sejam de fato executadas.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    device=0,
    patience=50,
    project="runs",
    name="epi-v1",
)

print("Pesos salvos em:", results.save_dir)


## 5. Métricas finais — mAP, precisão e recall

### Entregável: métricas finais de treinamento (mAP, precisão e recall)

In [ ]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
print("best.pt:", best_weights)

model = YOLO(str(best_weights))
metrics = model.val(data=str(data_yaml_path), split="val")

print()
print(f"mAP@0.5      = {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 = {metrics.box.map:.4f}")
print(f"Precisão     = {metrics.box.mp:.4f}")
print(f"Recall       = {metrics.box.mr:.4f}")


## 6. Imagem de resultado com bounding boxes

### Entregável: imagem de resultado da detecção com bounding boxes exportada como `.jpg`

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

val_images_dir = Path(data_cfg["path"]) / "valid" / "images"
amostra = sorted(glob.glob(str(val_images_dir / "*.jpg")))[0]
print("Imagem de teste:", amostra)

pred = model.predict(amostra, conf=0.3, save=True, project="runs", name="predict-epi")
saida_jpg = Path(pred[0].save_dir) / Path(amostra).name
print("Detecção salva em:", saida_jpg)

plt.figure(figsize=(10, 8))
plt.imshow(Image.open(saida_jpg))
plt.axis("off")
plt.title("Detecção YOLOv8n — epi-v1")
plt.show()


## 7. Baixando os artefatos (`best.pt`, imagem de detecção, curvas de treino)

In [ ]:
import shutil
from google.colab import files

shutil.copy(best_weights, "best.pt")
shutil.copy(saida_jpg, "deteccao_bounding_boxes.jpg")
shutil.copy(Path(results.save_dir) / "results.png", "results.png")
shutil.copy(data_yaml_path, "dataset.yaml")

for arquivo in ["best.pt", "deteccao_bounding_boxes.jpg", "results.png", "dataset.yaml"]:
    files.download(arquivo)


## Resumo dos entregáveis desta atividade

1. **Link deste notebook Colab** com YOLOv8n treinado por 50 épocas e métricas visíveis
   (compartilhar como "Qualquer pessoa com o link").
2. **Arquivo `best.pt`** gerado — baixado na célula 7.
3. **Métricas finais** (mAP@0.5, mAP@0.5:0.95, precisão, recall) — print da célula 5.
4. **Imagem `.jpg`** com bounding boxes — `deteccao_bounding_boxes.jpg`, baixado na célula 7.
5. **`dataset.yaml`** com caminhos de train/val e lista de classes — célula 3 / arquivo baixado.